# 03 — Ownership Path Exploration
Explore direct owners, full ownership chains, and ultimate beneficial owners (UBOs).

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository

settings = get_neo4j_settings()
repo = Neo4jRepository(**vars(settings))
repo.test_connection()
print("Connected")

Connected


## Target company
Set the company name to investigate. Use notebook 02 to find exact names.

In [3]:
COMPANY_NAME = "TELEFONICA O2 HOLDINGS LIMITED"
COMPANY_NAME = "ESBEN MOTORSPORT LIMITED"

## Direct owners (1 hop)

In [4]:
direct = repo.get_direct_owners(COMPANY_NAME)
print(f"Direct owners of '{COMPANY_NAME}': {len(direct)}")

if direct:
    df = pd.DataFrame(direct)
    display(df)
else:
    print("None found — check COMPANY_NAME above.")

Direct owners of 'ESBEN MOTORSPORT LIMITED': 1


,owner_name,owner_labels,ownership_pct_min,ownership_pct_max,ownership_controls
0,ESBEN BUSINESS IT SOLUTIONS LIMITED,[Company],75,100,[ownership-of-shares-75-to-100-percent]


## Ownership percentage breakdown
Inspect `ownership_pct_min` and `ownership_pct_max` for direct owners.

In [5]:
if direct:
    df_pct = pd.DataFrame(direct)[[
        "owner_name", "ownership_pct_min", "ownership_pct_max", "ownership_controls"
    ]]
    df_pct["pct_range"] = df_pct.apply(
        lambda r: f"{r.ownership_pct_min}–{r.ownership_pct_max}%"
        if pd.notna(r.ownership_pct_min) else "not specified",
        axis=1,
    )
    display(df_pct[["owner_name", "pct_range", "ownership_controls"]])

,owner_name,pct_range,ownership_controls
0,ESBEN BUSINESS IT SOLUTIONS LIMITED,75–100%,[ownership-of-shares-75-to-100-percent]


## Full ownership paths (multi-hop)
Each row is one hop in one path. `path_depth` = total hops in that path.

In [6]:
paths = repo.get_ownership_paths(COMPANY_NAME, max_depth=5, limit=50)
print(f"Path hops returned: {len(paths)}")

if paths:
    df_paths = pd.DataFrame(paths)
    display(df_paths)

Path hops returned: 5


,path_depth,hop,from_name,from_labels,ownership_pct_min,ownership_pct_max,ownership_controls,to_name,to_labels
0,1,1,ESBEN BUSINESS IT SOLUTIONS LIMITED,[Company],75,100,[ownership-of-shares-75-to-100-percent],ESBEN MOTORSPORT LIMITED,[Company]
1,2,1,None,[Individual],50,75,[ownership-of-shares-50-to-75-percent],ESBEN BUSINESS IT SOLUTIONS LIMITED,[Company]
2,2,1,None,[Individual],25,50,[ownership-of-shares-25-to-50-percent],ESBEN BUSINESS IT SOLUTIONS LIMITED,[Company]
3,2,2,ESBEN BUSINESS IT SOLUTIONS LIMITED,[Company],75,100,[ownership-of-shares-75-to-100-percent],ESBEN MOTORSPORT LIMITED,[Company]
4,2,2,ESBEN BUSINESS IT SOLUTIONS LIMITED,[Company],75,100,[ownership-of-shares-75-to-100-percent],ESBEN MOTORSPORT LIMITED,[Company]


### Paths grouped by depth

In [7]:
if paths:
    depth_summary = (
        df_paths.groupby("path_depth")["from_name"]
        .nunique()
        .reset_index()
        .rename(columns={"from_name": "unique_owners"})
    )
    display(depth_summary)

,path_depth,unique_owners
0,1,1
1,2,1


## Ultimate individual owners (UBOs)
Non-company leaf nodes at the top of every ownership chain.

In [8]:
ubos = repo.get_ultimate_individual_owners(COMPANY_NAME)
print(f"Ultimate individual owners: {len(ubos)}")

if ubos:
    df_ubos = pd.DataFrame(ubos)
    display(df_ubos)
else:
    print("None found — all ownership chains terminate at companies.")

Ultimate individual owners: 2


,owner_name,owner_labels,chain_depth,ownership_pct_min,ownership_pct_max,ownership_controls
0,None,[Individual],2,50,75,[ownership-of-shares-50-to-75-percent]
1,None,[Individual],2,25,50,[ownership-of-shares-25-to-50-percent]


## Close

In [9]:
repo.close()
print("Driver closed")

Driver closed
